In [ ]:
# =========================================================
# [0] 데이터 불러오기
#  - 0611데이터분석용.xlsx 안의 두 시트를 읽는다.
#      · '570' : MTT formazan 흡광도 (실제 신호)
#      · '630' : 참조 파장 흡광도 (배경/탁도 보정용)
#  - 분석에는 (570 - 630) 보정값을 쓴다. (셀 4 참고)
# =========================================================
import pandas as pd
from pathlib import Path

# 데이터 파일 경로 (노트북과 같은 폴더 = 저장소 최상위에 둔다)
excel_path = Path('0611데이터분석용.xlsx')

df_570 = pd.read_excel(excel_path, sheet_name='570')
df_630 = pd.read_excel(excel_path, sheet_name='630')

# 시트가 잘 읽혔는지 형태(행, 열) 확인  → (6, 11) 기대: well 6개(B~G) × (라벨 + 농도 10열)
print('df_570 shape:', df_570.shape)
print('df_630 shape:', df_630.shape)

df_570.head(), df_630.head()


In [ ]:
# =========================================================
# [1] 시트 정리(clean) 함수
#  - 96-well 플레이트의 가장자리(edge)는 증발 영향이 커서 보통 분석에서 뺀다.
#      · 행 'A','H' : 맨 위/아래 가장자리 행 → 제거
#      · 열 1, 12   : 양쪽 끝 열           → 제거
#  - 첫 열(라벨 B~G)을 index로 보내고, 빈칸(NaN) 있는 행은 버린다.
#  ※ 이 파일은 이미 B~G·농도열만 남긴 상태라 아래 drop은 대부분 '해당 없음'으로
#     그냥 통과한다(원본 플레이트 형식도 처리되도록 둔 안전장치).
# =========================================================
def clean_sheet(df):
    label_col = df.columns[0]                                   # 첫 열 = well 행 라벨(B~G)
    df = df[~df[label_col].isin(['A', 'H'])]                    # 가장자리 행 제거
    df = df.set_index(label_col)                                # 라벨을 index로
    drop_cols = [c for c in [1, '1', 12, '12'] if c in df.columns]  # 가장자리 열
    df = df.drop(columns=drop_cols)
    return df.dropna(how='any')                                 # 빈칸 있는 행 제거


df_570 = clean_sheet(df_570)
df_630 = clean_sheet(df_630)

print('df_570 shape:', df_570.shape)
print('df_630 shape:', df_630.shape)
df_570.head(), df_630.head()


In [ ]:
df_570   # 570 nm 흡광도 (index=well B~G, columns=농도)


In [ ]:
df_630   # 630 nm 참조 흡광도


In [ ]:
# =========================================================
# [4] 파장 보정: 570 nm 신호에서 630 nm(참조)를 빼서
#     배경/탁도/플레이트 바닥 영향을 상쇄 → 순수 formazan 신호로 사용
# =========================================================
mtt = df_570 - df_630


### 보정 신호(mtt) 확인 및 baseline(0 µM) 정리


In [ ]:
mtt   # (570-630) 보정값. 아직 0 µM 대조군이 2개 열로 들어있음


In [ ]:
# =========================================================
# [7] 0 µM(무처리 대조군) 열이 2개 → 1개만 남긴다
#  - 실험 설계상 0 µM 대조군을 2열(플레이트 C열, D열)로 찍었다.
#  - 첫 번째(C열)는 피펫팅 편차가 커서 baseline으로 부적절하므로,
#    iloc[:, 1:] 로 '첫 0 µM 열' 하나만 삭제하고 두 번째(D열)만 baseline으로 사용.
#  - 남은 9개 열에 사람이 읽기 쉬운 농도 이름을 직접 붙인다.
# =========================================================
mtt = mtt.iloc[:, 1:]   # 맨 앞 열(= 첫 번째 0 µM) 제거 → 0 µM이 1개만 남음

mtt.columns = ['0μM', '100μM', '150μM', '200μM',
               '250μM', '300μM', '400μM', '500μM', '1000μM']
print("컬럼:", list(mtt.columns))
mtt


In [ ]:
mtt   # 0 µM 1열 + 처리 농도 8열, well 6개(B~G)


In [ ]:
# =========================================================
# [9] 농도별 측정값을 {농도: [well 6개 값]} 딕셔너리로 변환
#     (이후 평균/표준편차 계산과 그래프 점 찍기에 사용)
# =========================================================
print("mtt shape:", mtt.shape)
print("mtt columns:", list(mtt.columns))
mtt_dict = {}
for col in mtt.columns.unique():
    subset = mtt.loc[:, col]
    if isinstance(subset, pd.DataFrame):     # 같은 이름 열이 중복돼도 안전하게 평탄화
        mtt_dict[col] = subset.values.flatten().tolist()
    else:
        mtt_dict[col] = subset.tolist()
mtt_dict


In [ ]:
# =========================================================
# [10] 세포 생존율(%) 계산
#  - 0 µM(무처리)을 100% 기준으로 본다.
#  - 같은 well 행끼리 나눈다(행 단위 정규화):
#        생존율 = (각 농도 흡광도 / 그 행의 0 µM 흡광도) × 100
#  - 따라서 0 µM 열은 모두 100%가 된다.
# =========================================================
baseline = mtt['0μM']   # index(B~G)로 정렬되어 행 단위로 나눠짐

viability_dict = {}
for conc in mtt.columns:
    viability_dict[conc] = (mtt[conc] / baseline * 100).tolist()

viability_dict


In [ ]:
# =========================================================
# [11] 4PL 회귀로 IC50 산출 + 그래프
#  - 0 µM 제외, 농도별 개별 well 값(6개씩)을 모두 펼쳐 한 번에 피팅.
#  - 결과: IC50, Hill, 95% CI 출력 + dose-response 곡선 그림 저장.
# =========================================================
# lmfit 기반 IC50 분석 (4PL 억제 곡선 + 95% 신뢰구간)
from lmfit import Model
from matplotlib.ticker import NullLocator
from matplotlib.transforms import blended_transform_factory
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")  # lmfit 신뢰구간 경고 등 콘솔에 로컬 경로가 찍히는 것 방지

# 0μM 제외, 농도별 개별 측정값 펼치기
conc_labels = ['100μM', '150μM', '200μM', '250μM', '300μM', '400μM', '500μM', '1000μM']
conc_values = np.array([100.0, 150.0, 200.0, 250.0, 300.0, 400.0, 500.0, 1000.0])
x_all, y_all = [], []
for label, num in zip(conc_labels, conc_values):
    vals = viability_dict[label]
    x_all.extend([num] * len(vals))
    y_all.extend(vals)

x_all = np.array(x_all)
y_all = np.array(y_all)

# 4PL(4-모수 로지스틱) 억제 곡선 정의
#   top    : 상단 평탄값(무처리 생존율, %)
#   bottom : 하단 평탄값(고농도에서 남는 생존율, %)
#   ic50   : 생존율이 top-bottom의 절반이 되는 농도
#   hill   : 곡선의 기울기(가파른 정도)
def four_pl(x, top, bottom, ic50, hill):
    return bottom + (top - bottom) / (1 + (x / ic50) ** hill)

model = Model(four_pl)
params = model.make_params(top=100, bottom=5, ic50=100, hill=1.5)  # 초기값
# ── 제약 조건: 데이터에 위·아래 평탄구간이 뚜렷하지 않으면 4PL이 발산하므로 묶어줌 ──
params['top'].set(vary=False)        # 0 µM=100% 기준이므로 상단은 100 고정
params['bottom'].set(min=0, max=30)  # 고농도 바닥 생존율: 0~30% 범위로 제한
params['ic50'].set(min=1, max=10000) # IC50 탐색 범위(µM)
params['hill'].set(min=0.1, max=10)  # 기울기 탐색 범위

result = model.fit(y_all, params, x=x_all)

# 95% 신뢰구간 (conf_interval은 ±1,2,3σ 표를 반환 → [1]=하한, [5]=상한 행의 값)
ci = result.conf_interval()
ic50_val  = result.params['ic50'].value
ic50_low  = ci['ic50'][1][1]   # 약 95% 하한
ic50_high = ci['ic50'][5][1]   # 약 95% 상한

print(f"IC50       : {ic50_val:.3f} μM  (95% CI: {ic50_low:.2f} – {ic50_high:.2f})")
print(f"Hill slope : {result.params['hill'].value:.4f}")
print(f"Top        : {result.params['top'].value:.2f} % (고정)")
print(f"Bottom     : {result.params['bottom'].value:.2f} %")
print(f"R²         : {1 - result.residual.var() / np.var(y_all):.4f}")

# 시각화
fig, ax = plt.subplots(figsize=(10, 6))

# 0μM은 로그 스케일에서 x=10 위치에 배치 (관례적 nominal 위치)
ZERO_X = 10.0
all_labels  = ['0μM'] + conc_labels
all_x_vals  = np.concatenate([[ZERO_X], conc_values])

rng = np.random.default_rng(0)
for num, label in zip(all_x_vals, all_labels):
    vals = np.array(viability_dict[label])
    jitter = rng.uniform(-0.04, 0.04, size=len(vals))
    ax.scatter(num * 10**jitter, vals, color='steelblue', alpha=0.65, s=45, zorder=4)

means = [np.mean(viability_dict[l]) for l in all_labels]
stds  = [np.std(viability_dict[l])  for l in all_labels]
ax.errorbar(all_x_vals, means, yerr=stds,
            fmt='D', color='black', ecolor='black',
            elinewidth=2, capsize=6, capthick=2,
            markersize=7, zorder=6, label='Mean ± SD')

x_fit = np.logspace(np.log10(ZERO_X) - 0.1,
                    np.log10(conc_values[-1]) + 0.5, 300)
y_fit = four_pl(x_fit, **{k: result.params[k].value for k in ['top','bottom','ic50','hill']})
ax.plot(x_fit, y_fit, color='crimson', linewidth=3,
        label=f"4PL Fit (Hill = {result.params['hill'].value:.2f})", zorder=3)

ax.axvline(ic50_val, color='forestgreen', linestyle='--', linewidth=2,
           label=f"IC50 = {ic50_val:.1f} μM", zorder=2)
ax.axvspan(ic50_low, ic50_high, alpha=0.15, color='forestgreen', label='95% CI')
ax.text(ic50_val * 1.06, result.params['bottom'].value + 5,
        f"{ic50_val:.1f} μM", color='forestgreen', fontsize=10)

# X축: 교차 배치 (staggered labels), 0μM은 ZERO_X 위치에 표시
ax.set_xscale('log')
exp_ticks  = [100, 150, 200, 250, 300, 400, 500, 1000]
all_ticks  = sorted(set([ZERO_X] + exp_ticks))   # [10, 100, 150, ..., 1000]

tick_labels = []
for t in all_ticks:
    if t == ZERO_X:
        tick_labels.append('0 μM')
    else:
        tick_labels.append(f'{int(t)} μM')

ax.set_xticks(all_ticks)
ax.set_xticklabels([''] * len(all_ticks))
ax.xaxis.set_minor_locator(NullLocator())


ax.set_xlabel('Concentration (μM)', fontsize=8, fontweight='bold', labelpad=32)
ax.set_ylabel('Cell Viability (%)', fontsize=8, fontweight='bold')
ax.set_title('H₂O₂-induced cytotoxicity in HEK293T cells (MTT assay)', fontsize=10, fontweight='bold', pad=50)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10, loc='upper right')

fig.subplots_adjust(bottom=0.22)

# 결과 그래프 저장 (저장소 최상위, README가 보여주는 파일과 같은 이름)
save_path = Path('MTT assay.png')
fig.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"저장 완료: {save_path}")   # 상대경로만 출력 (절대경로/.resolve 미사용 → 계정명 노출 방지)

plt.show()

### 필요 패키지
터미널에서:
```
pip install -r requirements.txt
```
(개별 설치) `pip install lmfit pandas numpy matplotlib openpyxl`
